# Compare original and fine-tuned Magi
No training or saved files. Recompute actual crop embeddings for both systems.
Use the same held-out pairs, detections, page-local CL clusters, and unchanged
CL→S clustering functions from `testMLP.ipynb`. Unlabeled appearances participate
in clustering but never supply supervised labels.

Pair thresholds are the cross-page validation thresholds saved during training.
Clustering thresholds are selected independently on validation, using the existing
0–1 grid. Test results never choose thresholds. Labels remain OCR/link-derived
supervision, rather than manually verified identities.

Supervised pairs and example labels exclude `NARRATION` and `UNKNOWN`; their
crops remain in clustering. Rebuild data and retrain before evaluating this version.


In [11]:
import torch
import torch.nn.functional as F
import numpy as np
import hashlib, random
from pathlib import Path
from PIL import Image

ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/fineTuningMagiCharSimHeads')
DATA_FILE = ROOT / 'training_data.pt'
MODEL_FILE = ROOT / 'magi_crop_last_block.pt'
BASE_MODEL = Path('C:/MODELS/magiv2')

def file_hash(path):
    with Path(path).open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()

import pandas as pd
from IPython.display import display

ENCODE_BATCH_SIZE = 32
assert torch.cuda.is_available(), 'Run with CUDA-enabled PyTorch.'
device = torch.device('cuda:0')
data = torch.load(DATA_FILE, map_location='cpu', weights_only=True)
assert MODEL_FILE.is_file(), 'Run all cells of trainMagiCropEncoder.ipynb first to save the checkpoint.'
saved = torch.load(MODEL_FILE, map_location='cpu', weights_only=True)
assert data['version'] == saved['version'] == 2, 'Rebuild data and rerun training with the updated notebooks.'
assert saved['settings']['threshold_selection'] == 'midpoint'
assert file_hash(DATA_FILE) == saved['dataset_hash'], 'Dataset changed since training.'
for name, expected in saved['base_model_hashes'].items():
    assert file_hash(BASE_MODEL / name) == expected, f'Original model changed: {name}'
print('Selected epoch:', saved['best_epoch'])


Selected epoch: 16


## Encode all validation/test appearances for each system


In [12]:
from transformers import AutoConfig, AutoModel
from transformers.utils import logging as transformer_logging

# Skip unused OCR/detection modules; the original crop weights and processor stay identical.
config = AutoConfig.from_pretrained(str(BASE_MODEL), trust_remote_code=True, local_files_only=True)
config.disable_ocr = True
config.disable_detections = True
# The omitted modules produce a long expected warning; check loading details ourselves.
logger = transformer_logging.get_logger('transformers.modeling_utils')
old_level = logger.level
logger.setLevel('ERROR')
try:
    model, loading = AutoModel.from_pretrained(str(BASE_MODEL), config=config, trust_remote_code=True,
        local_files_only=True, output_loading_info=True)
finally:
    logger.setLevel(old_level)
assert not loading['missing_keys'] and not loading['mismatched_keys'] and not loading['error_msgs']
assert not any(name.startswith('crop_embedding_model.') for name in loading['unexpected_keys'])
encoder = model.crop_embedding_model
encoder.embeddings.config.mask_ratio = 0.0  # Magi disables MAE masking for crop embeddings.
assert encoder.config.hidden_size == 768
encoder.to(device)
model.eval()

def load_crop(index):
    record = data['appearances'][index]
    with Image.open(record['page_path']) as image:
        page = np.asarray(image.convert('RGB'))
    return model.processor.crop_image(page, [record['bbox']])[0]

def encode_crops(crops):
    pixels = model.processor.preprocess_inputs_for_crop_embeddings(crops).to(device)
    output = encoder(pixels).last_hidden_state[:, 0]
    return F.normalize(output, dim=-1)

@torch.inference_mode()
def encode_appearances(indices):
    encoder.eval()
    embeddings = {}
    for start in range(0, len(indices), ENCODE_BATCH_SIZE):
        batch = indices[start:start + ENCODE_BATCH_SIZE]
        vectors = encode_crops([load_crop(index) for index in batch]).cpu()
        embeddings.update(zip(batch, vectors))
    return embeddings

indices = [index for i in data['splits']['validation'] + data['splits']['test']
           for index in data['sequence_rows'][i]]

def sequence_features(embeddings):
    return {i: torch.stack([embeddings[row] for row in data['sequence_rows'][i]]).numpy()
            if data['sequence_rows'][i] else np.empty((0, 768), dtype=np.float32)
            for i in data['splits']['validation'] + data['splits']['test']}

features = {'Original Magi': sequence_features(encode_appearances(indices))}
prefix = f'crop_embedding_model.encoder.layer.{len(encoder.encoder.layer)-1}.'
parameters = dict(model.named_parameters())
expected_names = {name for name in parameters if name.startswith(prefix)}
assert saved['trainable_prefix'] == prefix and set(saved['state_dict']) == expected_names
with torch.no_grad():
    for name, tensor in saved['state_dict'].items():
        assert tensor.shape == parameters[name].shape
        parameters[name].copy_(tensor.to(device))
features['Fine-tuned Magi'] = sequence_features(encode_appearances(indices))


## Test pair classification using saved validation thresholds


In [13]:
def pair_scores(split, vectors):
    rows = data['pairs'][split].tolist()
    scores = np.array([np.dot(vectors[i][a], vectors[i][b]) for i, a, b, _ in rows]).clip(-1, 1)
    labels = np.array([y for _, _, _, y in rows])
    cross = np.array([int(data['sequences'][i]['page'][a]) != int(data['sequences'][i]['page'][b])
                      for i, a, b, _ in rows])
    assert set(labels.tolist()) == {0, 1}
    return scores, labels, cross

def metrics(scores, labels, threshold):
    predicted = np.asarray(scores, dtype=np.float64) >= threshold
    positive, negative = labels == 1, labels == 0
    same = float(predicted[positive].mean()) if positive.any() else float('nan')
    different = float((~predicted[negative]).mean()) if negative.any() else float('nan')
    return dict(balanced_accuracy=(same + different) / 2,
                same_recall=same, different_recall=different,
                accuracy=float((predicted == labels).mean()) if len(labels) else float('nan'))

def choose_threshold(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)
    assert len(scores) == len(labels) and np.isfinite(scores).all()
    assert set(labels.tolist()) == {0, 1}
    order = np.argsort(-scores, kind='stable')
    values, targets = scores[order], labels[order]
    ends = np.r_[np.where(values[:-1] != values[1:])[0], len(values)-1]
    tp = np.cumsum(targets)[ends]
    fp = (ends + 1) - tp
    balanced = (tp / targets.sum() + 1 - fp / (len(targets)-targets.sum())) / 2
    best = int(np.argmax(balanced))
    if balanced[best] <= 0.5:
        # Use the empty SAME interval above the largest cosine score.
        return float((values[0] + 1.0) / 2) if values[0] < 1.0 else 1.0 + 1e-6
    upper = float(values[ends[best]])
    lower = float(values[ends[best] + 1]) if ends[best] + 1 < len(values) else -1.0
    return lower + (upper - lower) / 2  # Midpoint avoids placing the threshold on an observed score.


pair_thresholds = {'Original Magi': saved['baseline_validation']['threshold'],
                   'Fine-tuned Magi': saved['validation']['threshold']}
pair_results = []
for name, vectors in features.items():
    threshold = pair_thresholds[name]
    scores, labels, cross = pair_scores('validation', vectors)
    validation = metrics(scores[cross], labels[cross], threshold)
    print(name, '| rechecked validation cross-BA:', round(validation['balanced_accuracy'], 4))
    scores, labels, cross = pair_scores('test', vectors)
    pair_results.append(dict(method=name, validation_threshold=threshold,
                             **metrics(scores[cross], labels[cross], threshold)))
display(pd.DataFrame(pair_results).set_index('method').round(4))
print('Test cross-page pairs:', int(cross.sum()))


Original Magi | rechecked validation cross-BA: 0.7667
Fine-tuned Magi | rechecked validation cross-BA: 0.7976


,validation_threshold,balanced_accuracy,same_recall,different_recall,accuracy
method,,,,,
Original Magi,0.6727,0.7576,0.7056,0.8096,0.7793
Fine-tuned Magi,0.6978,0.7506,0.7273,0.7740,0.7604


Test cross-page pairs: 793


## Unchanged CL→S clustering and its evaluation


In [14]:
def cluster_entities(sequence, vectors, threshold):
    assignments = np.full(len(vectors), -1, dtype=int)
    entities = []
    pages = sequence['page'].numpy()
    labels = sequence['cluster'].numpy()
    characters = sequence['character'].numpy()
    for page in range(len(sequence['page_files'])):
        groups = {}
        for row in np.where(pages == page)[0]:
            key = int(labels[row]) if labels[row] >= 0 else -int(characters[row])-1
            groups.setdefault(key, []).append(int(row))
        candidates = []
        for key, members in groups.items():
            for entity, existing in enumerate(entities):
                score = float((vectors[members] @ vectors[existing].T).max())
                if score >= threshold:
                    candidates.append((score, key, entity))
        candidates.sort(key=lambda item: item[0], reverse=True)
        accepted, used_entities = {}, set()
        for score, key, entity in candidates:
            if key not in accepted and entity not in used_entities:
                accepted[key] = entity
                used_entities.add(entity)
        for key, members in groups.items():
            if key in accepted:
                entity = accepted[key]
            else:
                entity = len(entities)
                entities.append([])
            entities[entity].extend(members)
            assignments[members] = entity
    assert np.all(assignments >= 0)
    return assignments

def clustering_metrics(split, vectors, threshold):
    predicted = {i: cluster_entities(data['sequences'][i], vectors[i], threshold)
                 for i in data['splits'][split]}
    truth, same_entity = [], []
    for i, a, b, y in data['pairs'][split].tolist():
        s = data['sequences'][i]
        if s['page'][a] != s['page'][b]:
            truth.append(y)
            same_entity.append(int(predicted[i][a] == predicted[i][b]))
    assert truth, 'No labeled cross-page pairs for clustering evaluation.'
    result = metrics(np.array(same_entity), np.array(truth), 0.5)
    return dict(cross_page_pairs=len(truth), **result)


In [15]:
cluster_results, cluster_thresholds, assignments = [], {}, {}
for name, vectors in features.items():
    trials = []
    for threshold in np.linspace(0, 1, 51):
        result = clustering_metrics('validation', vectors, float(threshold))
        if np.isfinite(result['balanced_accuracy']):
            trials.append((result['balanced_accuracy'], float(threshold)))
    assert trials, 'Validation clustering needs both cross-page pair classes.'
    _, threshold = max(trials, key=lambda item: item[0])
    cluster_thresholds[name] = threshold
    cluster_results.append(dict(method=name, validation_threshold=threshold,
        **clustering_metrics('test', vectors, threshold)))
    assignments[name] = {i: cluster_entities(data['sequences'][i], vectors[i], threshold)
                         for i in data['splits']['test']}
display(pd.DataFrame(cluster_results).set_index('method').round(4))


,validation_threshold,cross_page_pairs,balanced_accuracy,same_recall,different_recall,accuracy
method,,,,,,
Original Magi,0.78,793,0.7083,0.5714,0.8452,0.7654
Fine-tuned Magi,0.84,793,0.7042,0.5455,0.8630,0.7705


## A few changed identities, including wrong merges
Show examples from accepted appearances. Entity numbers are local to each sequence;
compare membership, since renumbering alone does not mean an identity changed.


In [16]:
def show_sequence(i):
    sequence = data['sequences'][i]
    labeled = torch.where(sequence['gt'] >= 0)[0].tolist()
    print('\nSequence:', sequence['id'])
    for name in ['GT', 'Original Magi', 'Fine-tuned Magi']:
        groups = {}
        for row in labeled:
            key = sequence['speaker_names'][int(sequence['gt'][row])] if name == 'GT' else (
                f'S{int(assignments[name][i][row])+1}')
            appearance = data['appearances'][data['sequence_rows'][i][row]]['appearance_id']
            groups.setdefault(key, []).append(appearance)
        print(name, '|', ' ; '.join(f'{key} = {", ".join(members)}' for key, members in groups.items()))

improved, wrong_merges, other = [], [], []
for i in data['splits']['test']:
    original, tuned = assignments['Original Magi'][i], assignments['Fine-tuned Magi'][i]
    better = worse_merge = changed = False
    for seq, a, b, label in data['pairs']['test'].tolist():
        if seq != i or data['sequences'][i]['page'][a] == data['sequences'][i]['page'][b]:
            continue
        before, after = int(original[a] == original[b]), int(tuned[a] == tuned[b])
        changed |= before != after
        better |= before != label and after == label
        worse_merge |= label == 0 and before == 0 and after == 1
    if worse_merge:
        wrong_merges.append(i)
    elif better:
        improved.append(i)
    elif changed:
        other.append(i)
for title, examples in [('Improved examples', improved), ('New wrong merges', wrong_merges),
                         ('Other changes', other)]:
    print('\n' + title + ':', len(examples))
    for i in examples[:100]:
        show_sequence(i)
print('No files saved. Compare pair BA and clustering BA separately.')
print(len(improved))



Improved examples: 3

Sequence: seq_2e4b837ead6b88ae
GT | character1 = PG1:C1, PG1:C7, PG1:C8 ; character2 = PG1:C2, PG1:C6 ; character3 = PG2:C1, PG3:C2
Original Magi | S1 = PG1:C1 ; S2 = PG1:C2 ; S3 = PG1:C6, PG1:C7, PG1:C8, PG2:C1 ; S5 = PG3:C2
Fine-tuned Magi | S1 = PG1:C1 ; S2 = PG1:C2 ; S3 = PG1:C6, PG1:C7, PG1:C8 ; S4 = PG2:C1 ; S6 = PG3:C2

Sequence: seq_f1f5fb7a11b950f3
GT | character2 = PG1:C3, PG2:C10 ; character1 = PG1:C5, PG1:C8, PG1:C9, PG2:C5, PG2:C7, PG3:C1, PG3:C4
Original Magi | S3 = PG1:C3, PG1:C9, PG2:C7, PG2:C10, PG3:C1 ; S5 = PG1:C5, PG1:C8, PG2:C5 ; S1 = PG3:C4
Fine-tuned Magi | S3 = PG1:C3, PG1:C9, PG2:C7, PG2:C10 ; S5 = PG1:C5, PG1:C8, PG2:C5 ; S14 = PG3:C1 ; S9 = PG3:C4

Sequence: seq_864aaca43a3580bb
GT | character1 = PG1:C3, PG1:C7, PG1:C8, PG1:C10, PG1:C16 ; character3 = PG1:C9, PG2:C15, PG2:C16, PG3:C2, PG3:C10 ; character4 = PG2:C3, PG2:C13, PG3:C1, PG3:C4, PG3:C7 ; character5 = PG2:C5 ; character6 = PG2:C18, PG3:C3, PG3:C5, PG3:C8 ; character7 = PG3:C9
